# eutils2datasets — interactive examples

This notebook walks through the same examples as the README, plus a few extra edge cases. `eutils2datasets` never calls NCBI — it only *explains* what the [NCBI Datasets API v2](https://www.ncbi.nlm.nih.gov/datasets/docs/v2/) equivalent of an EUtils/EDirect call would be, where one exists.

No installation is required beyond the Python standard library. This notebook assumes it's running from the repo root (where this file lives, alongside the `eutils2datasets/` package directory); if you've `pip install -e .`'d the package, the `sys.path` line below is a no-op safety net.

In [ ]:
import sys, os

# Make the repo root importable, in case eutils2datasets isn't installed
repo_dir = os.getcwd()
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

from eutils2datasets import parse, translate, interpret


def show(result):
    """Pretty-print a TranslationResult the same way the CLI does."""
    if result.supported:
        print(f"Datasets equivalent found: {result.reason}")
        if result.datasets_cli:
            print(f"  CLI : {result.datasets_cli}")
        if result.datasets_rest:
            print(f"  REST: {result.datasets_rest}")
        if result.field_hint:
            print(f"  note: {result.field_hint}")
    else:
        print(f"No Datasets equivalent: {result.reason}")
        if result.eutils_reference:
            print(f"  Continue using EUtils/EDirect for this: {result.eutils_reference}")

## 1. Translating an EDirect pipeline

A gene lookup by symbol, restricted to human.

In [ ]:
req = parse('esearch -db gene -query "BRCA1[gene] AND human[orgn]"')
show(translate(req))

## 2. Translating a raw EUtils URL

An assembly lookup by accession.

In [ ]:
url = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=assembly&id=GCF_000001405.40'
req = parse(url)
show(translate(req))

## 3. Asking in plain language

`interpret()` runs a lightweight heuristic parser over a plain-language question, then feeds the result through the exact same `translate()` used above — so it can never give a different answer than the equivalent EUtils call would.

In [ ]:
show(interpret("find the BRCA1 gene in human"))

In [ ]:
show(interpret("what bioproject is GCF_000001405.40 from?"))

## 4. `elink`-style relationships

Some EUtils `elink` calls map to a Datasets *report field* rather than a separate link operation — e.g. an assembly's linked BioProject is just a field in the standard genome report, not a distinct call.

In [ ]:
req = parse(
    'elink -db assembly -target bioproject -id GCF_000001405.40'
)
show(translate(req))

## 5. When there's no Datasets equivalent

Datasets API v2 only covers gene, genome/assembly, taxonomy, and virus data. A PubMed search has no equivalent — the tool says so and hands back the original EUtils call for reference.

In [ ]:
req = parse('esearch -db pubmed -query "cancer"')
show(translate(req))

Boolean `OR`/`NOT`/grouping also has no structured Datasets equivalent, even within a supported database:

In [ ]:
req = parse('esearch -db gene -query "BRCA1[gene] OR BRCA2[gene]"')
show(translate(req))

## 6. Batch-translating a list of calls

In [ ]:
calls = [
    'efetch -db gene -id 672 -format docsum',
    'esearch -db taxonomy -query "mouse"',
    'esearch -db sra -query "RNA-seq human liver"',
]

for call in calls:
    print(f"> {call}")
    show(translate(parse(call)))
    print()

## 7. Inspecting the underlying data model

`parse()` returns a `ParsedRequest` and `translate()` returns a `TranslationResult` — both plain dataclasses, handy if you want to build something on top of this instead of using the CLI output directly.

In [ ]:
req = parse('esearch -db gene -query "BRCA1[gene] AND human[orgn]"')
print(req)
print()
result = translate(req)
print(result)

## 8. Complex real-world use cases

The examples above are all single-record lookups. The cells below cover patterns found in real production pipelines (Galaxy toolshed, nf-core, public-health genomics tools) that go beyond a one-line swap — see the README's "Complex real-world use cases" section for the full writeup and sources.

### Ortholog sets: the old elink is dead, not just outdated

EUtils' only cross-species ortholog link (`linkname=gene_gene_homolog`) was retired along with HomoloGene in 2019 and now silently returns zero results. Datasets' `--ortholog` flag is the only path that still works.

In [ ]:
req = parse('elink -db gene -target gene -id 672 -name gene_gene_homolog')
show(translate(req))
print()
# Also reachable via --ask, which lets you name taxa in plain language
show(interpret("orthologs of BRCA1 in mammals"))

### Bulk, filtered genome search — one call, not a loop

Picking "every complete, RefSeq genome for this species" used to mean an `esearch` against `db=assembly` followed by looping `esummary`/`efetch` over every hit to filter client-side. Datasets folds every filter into the same single taxon call.

In [ ]:
req = parse(
    'esearch -db assembly -query '
    '"Escherichia coli[orgn] AND \\"complete genome\\"[Assembly Level] AND srcdb_refseq[PROP]"'
)
show(translate(req))

### Taxonomic subtree traversal

`[Subtree]` is real EDirect syntax for "this taxon and everything under it" — used to build local reference databases (e.g. every species under a genus). Maps to Datasets' `--children`.

In [ ]:
req = parse('esearch -db taxonomy -query "9606[Subtree]"')
show(translate(req))

### Gene flanking-sequence extraction

`--include-flanks-bp` pads a gene download with surrounding genomic sequence, but only for `WP_` protein accessions. There's no EUtils flag for this at all, so it's `--ask`-only.

In [ ]:
show(interpret("100 bp flanking sequence for WP_003249567.1"))
print()
# Asking for flanking sequence around a non-WP_ accession is correctly rejected
show(interpret("100 bp flanking sequence for NP_000483.3"))